# Lab D - Perceptron implementing a NAND gate
A perceptron outputs `1 if w.x + b > 0 else 0` and learns with `w <- w + lr*(target - output)*x`. NAND is linearly separable, so it is learnable by one perceptron.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
X = np.array([[0,0],[0,1],[1,0],[1,1]]); y_nand = np.array([1,1,1,0]); y_xor = np.array([0,1,1,0])
step = lambda z: (z > 0).astype(int)

def train(X, y, lr=0.1, epochs=100, seed=0):
    rng = np.random.default_rng(seed); w = rng.normal(0, .1, 2); b = 0.0; hist = []
    for ep in range(epochs):
        errors = 0
        for xi, ti in zip(X, y):
            out = step(np.array([w @ xi + b]))[0]; err = ti - out
            w += lr * err * xi; b += lr * err; errors += abs(err)
        hist.append(errors)
        if errors == 0: break
    return w, b, hist

w, b, hist = train(X, y_nand)
print('learned w =', w, ' b =', round(b, 3), ' converged after', len(hist), 'epochs')
print(pd.DataFrame({'x1': X[:,0], 'x2': X[:,1], 'w.x+b': (X @ w + b).round(3), 'output': step(X @ w + b), 'NAND': y_nand}))

### Hand-set weights (for the viva)
w = (-1, -1), b = 1.5: (0,0)->1.5>0 ->1; (0,1)->0.5>0 ->1; (1,0)->0.5>0 ->1; (1,1)->-0.5<=0 ->0. This is exactly NAND.

In [ ]:
w2, b2 = np.array([-1., -1.]), 1.5; print('hand-set outputs:', step(X @ w2 + b2), '== NAND', y_nand)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(hist, marker='o'); ax[0].set_xlabel('epoch'); ax[0].set_ylabel('misclassifications'); ax[0].set_title('NAND training')
for c, m in ((0, 'o'), (1, 's')): ax[1].scatter(*X[y_nand==c].T, marker=m, s=120, label=f'NAND={c}')
xs = np.linspace(-.2, 1.2); ax[1].plot(xs, -(w[0]*xs + b)/w[1], 'k--', label='learned boundary'); ax[1].set_xlim(-.2, 1.2); ax[1].set_ylim(-.2, 1.2); ax[1].legend(); plt.tight_layout(); plt.show()

### XOR is NOT linearly separable: a single perceptron cannot learn it
This is why we need hidden layers (an ANN).

In [ ]:
w, b, hist = train(X, y_xor, epochs=200); print('XOR misclassifications in last epoch:', hist[-1], 'after', len(hist), 'epochs (never reaches 0)'); print('outputs', step(X @ w + b), 'target', y_xor)